# Pipeline 1 — SST-2 & GLUE Tasks

**Owner:** Christine  
**Branch:** `develop`

## What this notebook does
Fine-tunes both `bert-base-uncased` and `distilbert-base-uncased` on:
- **SST-2** — sentiment classification (main task from paper Table 1)
- **MNLI** — natural language inference (optional, if time allows)

Results are saved to `results/` for the final comparison notebook.

## Expected results (from paper Table 1)
| Model | SST-2 |
|-------|-------|
| BERT-base | 93.5% |
| DistilBERT | 91.3% |

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import sys
sys.path.append('../src')

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

from train import train_model
from evaluate import evaluate_model
from utils import print_model_info, measure_inference_speed, save_results, count_parameters, model_size_mb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5
SEED       = 42

torch.manual_seed(SEED)

In [ ]:
# ── Load SST-2 ─────────────────────────────────────────────────────────────
# SST-2 is a binary sentiment classification task (positive / negative)
raw = load_dataset('glue', 'sst2')
print(f"Train: {len(raw['train'])} | Validation: {len(raw['validation'])}")

In [ ]:
# ── Helper: tokenize + build dataloaders ──────────────────────────────────
def get_dataloaders(model_name, dataset, text_col='sentence'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch[text_col], truncation=True,
                         max_length=MAX_LENGTH, padding='max_length')

    tokenized = dataset.map(tokenize, batched=True)

    # DistilBERT has no token_type_ids
    cols = ['input_ids', 'attention_mask', 'label']
    if 'distilbert' not in model_name:
        cols.append('token_type_ids')

    tokenized.set_format('torch', columns=cols)
    tokenized = tokenized.rename_column('label', 'labels')

    train_loader = DataLoader(tokenized['train'],      batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(tokenized['validation'], batch_size=BATCH_SIZE)
    return train_loader, val_loader


def run_pipeline(model_name, dataset, task_name, text_col='sentence'):
    """Fine-tune a model, evaluate it, and save results."""
    print(f'\n{"="*50}')
    print(f'Model: {model_name} | Task: {task_name}')
    print(f'{"="*50}')

    train_loader, val_loader = get_dataloaders(model_name, dataset, text_col)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    print_model_info(model, model_name)

    history = train_model(model, train_loader, val_loader, DEVICE, epochs=EPOCHS, lr=LR)
    eval_results = evaluate_model(model, val_loader, DEVICE)
    speed = measure_inference_speed(model, val_loader, DEVICE)

    total_params, _ = count_parameters(model)
    results = {
        'model':            model_name,
        'task':             task_name,
        'val_accuracy':     eval_results['accuracy'],
        'total_parameters': total_params,
        'model_size_mb':    model_size_mb(model),
        'inference_speed':  speed,
        'training_history': history,
    }

    filename = f"{task_name}_{model_name.replace('/', '_').replace('-', '_')}.json"
    save_results(results, filename)
    print(f"Accuracy: {eval_results['accuracy']}%")
    return results

In [ ]:
# ── Run SST-2: BERT-base ───────────────────────────────────────────────────
bert_sst2 = run_pipeline('bert-base-uncased', raw, 'sst2')

In [ ]:
# ── Run SST-2: DistilBERT ──────────────────────────────────────────────────
distilbert_sst2 = run_pipeline('distilbert-base-uncased', raw, 'sst2')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────
import pandas as pd

paper_results = {
    'bert-base-uncased':       {'sst2': 93.5},
    'distilbert-base-uncased': {'sst2': 91.3},
}

rows = [
    {'Model': 'BERT-base (paper)',    'SST-2': paper_results['bert-base-uncased']['sst2']},
    {'Model': 'DistilBERT (paper)',   'SST-2': paper_results['distilbert-base-uncased']['sst2']},
    {'Model': 'BERT-base (ours)',     'SST-2': bert_sst2['val_accuracy']},
    {'Model': 'DistilBERT (ours)',    'SST-2': distilbert_sst2['val_accuracy']},
]

df = pd.DataFrame(rows)
print(df.to_string(index=False))